# Notebook 3: Cluster Profiling, Stability & Real-World Validation
## Road Segment Congestion Pattern Discovery with K-Means

### 1. Overview
In this notebook, we translate mathematical clusters into actionable municipal engineering insights:
1. **Cluster Profiling:** Mean and median statistics in original engineering units.
2. **Plain-Language Domain Personas:** Meaningful labels for operational planning.
3. **Unsupervised Validation:** Cross-tabulation against the held-out `Traffic Situation` label.
4. **Stability Analysis:** Pairwise Adjusted Rand Index (ARI) under multi-seed initialization and $80\%$ bootstrap subsampling.
5. **Limitations, Sampling Bias & Ethical Use Guidelines.**


In [ ]:
import sys
import os
sys.path.append(os.path.abspath("../src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans

from evaluation import (
    generate_cluster_profiles,
    cross_tabulate_traffic_situation,
    test_random_seed_stability,
    test_subsampling_stability
)

RANDOM_STATE = 42
df_clean = pd.read_csv("../data/processed/traffic_clean.csv")
X_scaled_df = pd.read_csv("../data/processed/X_scaled.csv")
X = X_scaled_df.values

# Fit chosen k=4 model
kmeans = KMeans(n_clusters=4, n_init=25, random_state=RANDOM_STATE)
df_clean["cluster"] = kmeans.fit_predict(X)
print("Fitted optimal k=4 K-Means model.")


### 2. Original-Unit Feature Profiles
We examine actual vehicle counts and ratios across clusters.


In [ ]:
profile_mean, profile_median = generate_cluster_profiles(df_clean, df_clean["cluster"].values)
print("Mean Cluster Profiles (Original Physical Units):")
profile_mean[["sample_count", "percentage", "total_volume", "CarCount", "BikeCount", "BusCount", "TruckCount", "hour_float", "is_weekend"]]


### 3. Standardized Centroids Heatmap
Standardized centroids allow direct visual comparison across heterogeneous dimensions (temporal, composition, volume).


In [ ]:
centroids_df = pd.DataFrame(kmeans.cluster_centers_, columns=X_scaled_df.columns, index=[f"Cluster {i}" for i in range(4)])

plt.figure(figsize=(10, 5))
sns.heatmap(centroids_df, annot=True, fmt=".2f", cmap="Spectral", center=0, linewidths=0.5)
plt.title("Standardized Cluster Centroids (Feature Deviation from Mean)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()


### 4. Plain-Language Operational Names
Based on empirical profiles, we assign operational definitions to each cluster:
- **Cluster 0: "Midday & Afternoon Commercial Flow"** (High traffic volume, balanced car/truck mix, weekday afternoons).
- **Cluster 1: "Peak Commuter Rush Hour"** (Highest total volume, car and bike surge, severe congestion, weekday mornings/evenings).
- **Cluster 2: "Nighttime / Early Dawn Low Flow"** (Low overall volume, high truck freight delivery ratio, quiet roads).
- **Cluster 3: "Weekend Leisure Traffic"** (Exclusively weekend intervals, moderate personal car/bike volume, minimal freight/transit).


### 5. Cross-Tabulation with Held-Out Ground Truth (`Traffic Situation`)
We cross-tabulate our unsupervised cluster assignments against the held-out `Traffic Situation` label.


In [ ]:
counts_ct, pct_ct = cross_tabulate_traffic_situation(df_clean["Traffic Situation"], df_clean["cluster"].values)
print("Percentage Composition of Traffic Situation by Cluster:")
pct_ct


In [ ]:
plt.figure(figsize=(9, 5))
order = [c for c in ["low", "normal", "high", "heavy"] if c in pct_ct.columns]
pct_ct[order].plot(kind="bar", stacked=True, colormap="RdYlGn_r", edgecolor="black")
plt.title("Held-Out 'Traffic Situation' Ground Truth Distribution per Cluster", fontsize=13, fontweight="bold")
plt.xlabel("Discovered Cluster")
plt.ylabel("Percentage (%)")
plt.legend(title="Traffic Situation", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()


### 6. Cluster Stability Analysis
#### 6.1 Multi-Seed Initialization Test (ARI)
We evaluate whether cluster boundaries remain stable across 15 different random seeds.


In [ ]:
seed_stats, seed_aris = test_random_seed_stability(X, k=4, num_seeds=15)
print(f"Random Seed Stability (k=4): Mean ARI = {seed_stats['mean_ari']:.4f} (Min = {seed_stats['min_ari']:.4f}, Std = {seed_stats['std_ari']:.4f})")


#### 6.2 80% Bootstrap Subsampling Test (ARI)
We test resilience to data perturbations across 20 iterations of 80% random subsamples.


In [ ]:
sub_stats, sub_aris = test_subsampling_stability(X, k=4, sample_ratio=0.8, n_iterations=20)
print(f"Subsampling Stability (k=4): Mean ARI = {sub_stats['mean_subsample_ari']:.4f} (Min = {sub_stats['min_subsample_ari']:.4f}, Std = {sub_stats['std_subsample_ari']:.4f})")


### 7. Limitations, Sampling Bias & Ethical Considerations

#### Methodological Limitations:
1. **Spherical Cluster Geometry:** K-Means assumes isotropic, convex clusters with equal variance. Real-world traffic dynamics often present continuous topological manifolds.
2. **Lack of Temporal Auto-Correlation Modeling:** K-Means treats each 15-minute time-slot independently, ignoring that time $t$ is strongly autocorrelated with $t-1$.
3. **Metric Sensitivity:** Scaling choices (StandardScaler vs. RobustScaler) significantly affect distance metrics.

#### Potential Sampling Biases:
1. **Single Sensor Location:** Recordings originate from a single monitoring station; insights may not generalize to arterial roads with different commercial profiles.
2. **Missing Modes:** Pedestrians, scooters, and non-motorized transport are absent from sensor logs.
3. **Seasonal and Weather Blindness:** The dataset spans a limited observation window without rainfall, snow, or holiday annotations.

#### Responsible and Ethical Decision Support:
1. **Human-in-the-Loop Governance:** Unsupervised clusters must serve as **decision support**, not automated enforcement or municipal resource cuts.
2. **Socioeconomic Equity:** Congestion patterns should not be used to disproportionately divert maintenance away from lower-income corridors.
3. **Privacy Considerations:** Aggregated vehicle counts at 15-minute resolution pose minimal re-identification risk, but linking with high-resolution plate cameras requires rigorous governance.
